## Model Exploration: Sameh

## Section 1 | Imports - Data Loading - Simple EDA
1.1  **Importing Libraries**

In [1]:
from os.path import sep
# Standard library
from pathlib import Path

# Data manipulation
import pandas as pd
import numpy as np

# Scikit-learn: preprocessing and pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Scikit-learn: model and evaluation
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, precision_recall_curve
)
from sklearn.calibration import calibration_curve

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


# Path setup
BASE_DIR = Path("../../")

DATA_PATH = BASE_DIR / "data" / "processed" / "sample.csv"

1.2 **Loading the Dataset**

In [2]:
# Load the dataset
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH, sep="|")
    print("Dataset loaded successfully.")
    print(f"Shape: ({df.shape[0]:,} rows, {df.shape[1]} columns)")
else:
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}")

Dataset loaded successfully.
Shape: (200,000 rows, 29 columns)


1.3 **Inspection of [sample data](../../data/processed/sample.csv)**

In [3]:
# Initial inspection of the processed sample before modeling.
# Display the first few rows of the dataset
print("First 5 rows of the dataset:")
display(df.head())

# Display summary statistics
print("\nSummary statistics:")
print(df.describe(include="all"))


# Display a list of columns and their data types
print("\nColumn names and data types:")
print(df.dtypes)

# Check for 'order' column distribution
print(f"\nTarget distribution (order):\n{df['order'].value_counts(normalize=True).map('{:.1%}'.format)}")


# Finally check for missing values
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

First 5 rows of the dataset:


,lineID,day,pid,adFlag,availability,competitorPrice,click,basket,order,price,...,rrp,has_competitor,campaignIndex_A,campaignIndex_B,campaignIndex_C,price_diff_competitor,price_ratio_competitor,price_pct_diff_competitor,is_post_shift_day,price_diff_vs_previous_available_day
0,978899,39,9624,0,1,17.19,1,0,0,19.89,...,21.51,1,0,0,0,2.70,1.1571,15.71,1,0.0
1,1267035,47,3969,1,1,18.13,1,0,0,20.85,...,26.07,1,0,0,0,2.72,1.1500,15.00,1,0.0
2,297914,14,16633,0,1,15.06,0,0,1,16.45,...,23.98,1,0,1,0,1.39,1.0923,9.23,0,0.0
3,2554963,87,20147,0,1,4.36,1,0,0,5.17,...,5.45,1,0,0,0,0.81,1.1858,18.58,1,0.0
4,2739211,92,14326,0,1,NaN,0,0,1,6.22,...,6.55,0,0,0,0,NaN,NaN,NaN,1,0.0



Summary statistics:
              lineID            day            pid         adFlag  \
count   2.000000e+05  200000.000000  200000.000000  200000.000000   
unique           NaN            NaN            NaN            NaN   
top              NaN            NaN            NaN            NaN   
freq             NaN            NaN            NaN            NaN   
mean    1.377866e+06      49.916220   10439.763875       0.317140   
std     7.954769e+05      25.292519    6448.469978       0.465364   
min     6.400000e+01       1.000000       3.000000       0.000000   
25%     6.892675e+05      31.000000    4315.000000       0.000000   
50%     1.377566e+06      50.000000   10057.000000       0.000000   
75%     2.066244e+06      71.000000   16138.000000       1.000000   
max     2.756002e+06      92.000000   22035.000000       1.000000   

        availability  competitorPrice          click         basket  \
count   200000.00000    192805.000000  200000.000000  200000.000000   
unique  

1.4 **Further inspection while adding the insights**

In [4]:
# Drop nulls (first-day-per-product rows have no previous price)
check_price_diff_vs_previous_available_day = df["price_diff_vs_previous_available_day"].dropna()

# Count how many rows have exactly zero price change
print(f"Non-null values: {len(check_price_diff_vs_previous_available_day):,}")
print(f"Zero values: {(check_price_diff_vs_previous_available_day == 0).sum():,} ({(check_price_diff_vs_previous_available_day == 0).mean():.1%})")

# Show full distribution summary
print(f"\nDistribution:")
print(check_price_diff_vs_previous_available_day.describe().to_string())

Non-null values: 197,347
Zero values: 170,607 (86.5%)

Distribution:
count    197347.000000
mean         -0.006322
std           1.137062
min         -46.050000
25%           0.000000
50%           0.000000
75%           0.000000
max          58.650000


## Data Inspection Insights

**Target distribution:** 74.4% not purchased, 25.6% purchased. This is moderate imbalance,
not extreme. The minority class has ~51,000 rows in this 200k sample (200,000 × 0.256),
which is more than enough for the model to learn from. I will use `class_weight="balanced"`
in the classifier and stratified splitting rather than resampling techniques.

**Missing values:** Six columns show missing values, but they trace back to two root causes:
- `competitorPrice` (7,195 missing, 3.6% of rows) propagates to `price_diff_competitor`,
  `price_ratio_competitor`, and `price_pct_diff_competitor` since those are derived from it.
  This is one gap, not four.
  - I think these three derived features been good for exploratory analysis and understanding the data,
    but they are redundant for modeling that might produce noise. But meanwhile `has_competitor_price` might compensate that loss of signal (We will see)
- `category` (6,290 missing, 3.1% of rows) represents products with no assigned shop
  category.
- `price_diff_vs_previous_available_day` (2,653 missing, 1.3% of rows) likely represents
  the first observation for a product where no previous day exists. as mentioned in [def price_diff_previous_available_day(df: pd.DataFrame) -> pd.DataFrame:](../../scripts/utils.py)
  - `price_diff_vs_previous_available_day` is relevant business information knowing customers' behavior, but seeing its distribution and the fact the median is 0.00, the 25th percentile is 0.00, the 75th percentile is 0.00. The vast majority of rows have zero price change day to day. So the feature is mostly flat with occasional spikes. That's low variance, which limits how much a model can learn from it. That's a strong argument for dropping it, especially since the missingness is not random (first observation for each product). However, I will keep it to measure its impact via feature importance. If it shows low importance, I will drop it.

**Columns to drop (data leakage):**
- `lineID`: unique row identifier, no predictive signal
- `revenue`: nonzero only when `order = 1`, directly encodes the target
- `click` and `basket`: mutually exclusive with `order`. In this dataset, each row records
  exactly one action (click, basket, or order). These are not behavioral history features, but rather alternative outcomes. Including them would leak the target. For example, if `click = 1`, then `order` must be 0, and vice versa. They are perfectly negatively correlated with `order` and do not represent independent signals. They are not features that describe user behavior leading up to a purchase, but rather labels of what action was taken. Therefore, they should be dropped to prevent data leakage.

**Columns we keep that could be questioned:**
- `day`: granular temporal feature. Risk of overfitting to sample-specific daily patterns,
  but I keep it to measure its impact via feature importance.
- `price` and `rrp`: correlated with `price_ratio_competitor` and other derived features,
  but may carry independent signal (absolute price sensitivity). I Keep them for now.
- `pid`: high cardinality (21,928 unique products). Encodes product identity, which bundles
  many latent attributes. Useful for tree-based models that can partition on it. Would be
  problematic for linear models.
- `content` and `unit`: I am not sure which value they could add to the model, but I will keep them for now to measure their impact via feature importance. If they show low importance, I will drop them.